## Buat Function untuk Proses Data

In [52]:
import pandas as pd
import numpy as np

def process_shipments(df):
    results = []
    seed = 0
    last_shipment_per_distributor = {}
    
    for distributor, group in df.groupby("Ship to"):
        while True:
            # Mengelompokkan berdasarkan Sales Document
            grouped = [grouping for _, grouping in group.groupby('Sales Document')]

            # Mengacak urutan kelompok
            np.random.seed(seed)
            np.random.shuffle(grouped)

            # Menggabungkan kembali ke DataFrame
            shuffled_df = pd.concat(grouped).reset_index(drop=True)
            
            shipments = []
            shipment_id = 1
            current_shipment = []
            current_utilization = 0
            temp_results = []
            low_utilization_shipments = []
            notifications = []

            split_rows = []  # Menyimpan baris yang butuh split DO

            prev_sales_doc = None  # Menyimpan Sales Document sebelumnya
            total_utilization_per_doc = shuffled_df.groupby("Sales Document")["Sum of Utilization"].sum().to_dict()
            
            for _, row in shuffled_df.iterrows():
                utilization = row["Sum of Utilization"]
                sales_doc = row["Sales Document"]
                
                if utilization > 0.999:
                    #notifications.append(f"Sales document: {sales_doc} pada delivery: {row['Cust. PO Number']} dengan utilization: {utilization:.4f} butuh split DO")
                    split_rows.append(row.to_dict())  # Simpan terpisah agar tidak masuk ke shipment biasa
                    continue  # Lewati iterasi untuk baris ini

                if prev_sales_doc and prev_sales_doc != sales_doc and (current_utilization + total_utilization_per_doc[sales_doc]) > 0.999:
                    # Jika melebihi batas atas, buat shipment baru, simpan shipment sebelumnya dan reset
                    if current_utilization < 0.7:
                        low_utilization_shipments.append(shipment_id)
                    
                    temp_results.extend([{**r, "Shipment": shipment_id} for r in current_shipment])
                    shipments.append(shipment_id)
                    shipment_id += 1
                    current_shipment = []
                    current_utilization = 0
                
                # Tambahkan baris ke shipment saat ini
                current_shipment.append(row.to_dict())
                current_utilization += utilization
                prev_sales_doc = sales_doc  # Perbarui Sales Document sebelumnya
            
            # Simpan shipment terakhir
            if current_shipment:
                temp_results.extend([{**r, "Shipment": shipment_id} for r in current_shipment])
                # Simpan shipment terakhir per distributor
                last_shipment_per_distributor[distributor] = {
                    "sales_doc": current_shipment[-1]['Sales Document'],
                    "po_number": current_shipment[-1]['Cust. PO Number'],
                    "utilization": current_utilization}
                # Pastikan shipment terakhir tidak dihitung dalam low utilization check
                if shipment_id in low_utilization_shipments:
                    low_utilization_shipments.remove(shipment_id)
            
            if not low_utilization_shipments: 
                results.extend(temp_results)
                break  # Keluar dari loop jika tidak ada shipment selain yang terakhir dengan utilization < 0.7
            else:
                seed += 1
        
        # Menambahkan shipment baru untuk baris yang butuh split DO
        for row in split_rows:
            shipment_id += 1
            results.append({**row, "Shipment": shipment_id})
            # Masukkan ke last_shipment_per_distributor agar dapat ditangani notifikasinya
            last_shipment_per_distributor[distributor] = {
                "sales_doc": row['Sales Document'],
                "po_number": row['Cust. PO Number'],
                "utilization": row['Sum of Utilization']
            }
                
    # Tambahkan shipment terakhir dari setiap distributor ke notifications
    for distributor, last_shipment in last_shipment_per_distributor.items():
        if last_shipment['utilization'] < 0.7:
            message = (f"Distributor: {distributor} - Sales document: {last_shipment['sales_doc']} pada delivery: {last_shipment['po_number']} terdapat sisa utilization sebesar {last_shipment['utilization']:.4f} yang tidak digunakan.")
            notifications.append(message)
        if last_shipment['utilization'] > 0.999:
            message = (f"Distributor: {distributor} - Sales document: {last_shipment['sales_doc']} pada delivery: {last_shipment['po_number']} dengan utilization: {last_shipment['utilization']:.4f} butuh split DO")
            notifications.append(message)
    return pd.DataFrame(results), notifications


## Buat Function untuk Import dan Export Input dan Output

In [53]:
# Proses dataset
def process_file(filepath, sheet_name, output_path):
    df = pd.read_excel(filepath, sheet_name=sheet_name)
    shipment_df, warnings = process_shipments(df)
    print(shipment_df.groupby(["Ship to", "Shipment"])["Sum of Utilization"].sum().reset_index())
    shipment_df.to_excel(output_path, index=False)
    for warning in warnings:
        print(warning)

## Implementasi untuk Data Example 1

In [54]:
# Eksekusi
process_file("D:/Kerjaan/Project P70960125 (Processing Data in Python)/Example 1.xlsx", "Input", "D:/Kerjaan/Project P70960125 (Processing Data in Python)/processed_shipments 1.xlsx")

     Ship to  Shipment  Sum of Utilization
0   15141048         1            0.897939
1   15141048         2            0.943568
2   15141048         3            0.808731
3   15141048         4            0.859514
4   15141048         5            0.705387
5   15141048         6            0.947458
6   15141048         7            0.210327
7   15228189         1            0.773464
8   15228189         2            0.689470
9   91418001         1            0.848468
10  91418001         2            0.843819
11  91418001         3            0.924851
12  91418001         4            0.897599
13  91418001         5            0.847065
14  91418001         6            0.743039
15  91418001         7            0.881457
16  91418001         8            0.975477
17  91418001         9            0.720902
18  91418001        10            0.887689
19  91418001        11            0.814249
Distributor: 15141048 - Sales document: 1024720666 pada delivery: SAGBT-04WT50001 terdapat sisa u

## Implementasi untuk Data Example 2

In [55]:
process_file("D:/Kerjaan/Project P70960125 (Processing Data in Python)/Example 2.xlsx", "Input", "D:/Kerjaan/Project P70960125 (Processing Data in Python)/processed_shipments 2.xlsx")

     Ship to  Shipment  Sum of Utilization
0   15141048         1            0.888051
1   15141048         2            0.778938
2   15141048         3            0.795923
3   15141048         4            0.758664
4   15141048         5            0.380943
5   15228189         1            0.901573
6   15228189         2            0.968171
7   15228189         3            0.138551
8   91418001         1            0.958577
9   91418001         2            0.938597
10  91418001         3            0.822277
11  91418001         4            0.913793
12  91418001         5            0.897517
13  91418001         6            0.743695
14  91418001         7            0.816977
15  91418001         8            0.619539
Distributor: 15141048 - Sales document: 1024652907 pada delivery: SAGBT-02H050014 terdapat sisa utilization sebesar 0.3809 yang tidak digunakan.
Distributor: 15228189 - Sales document: 1024653032 pada delivery: SAGBT-02F050005 terdapat sisa utilization sebesar 0.1386 y

## Implementasi untuk Data Example 3

In [56]:
process_file("D:/Kerjaan/Project P70960125 (Processing Data in Python)/Example 3.xlsx", "Input", "D:/Kerjaan/Project P70960125 (Processing Data in Python)/processed_shipments 3.xlsx")

     Ship to  Shipment  Sum of Utilization
0   15141048         1            0.953313
1   15141048         2            0.856290
2   15141048         3            0.971762
3   15141048         4            0.666370
4   15228189         1            0.959107
5   15228189         2            1.020000
6   91418001         1            0.725609
7   91418001         2            0.970379
8   91418001         3            0.912529
9   91418001         4            0.928417
10  91418001         5            0.955021
11  91418001         6            0.576533
Distributor: 15141048 - Sales document: 1024683088 pada delivery: SAGBT-03F050016 terdapat sisa utilization sebesar 0.6664 yang tidak digunakan.
Distributor: 15228189 - Sales document: 1024683097 pada delivery: SAGBT-03F050014 dengan utilization: 1.0200 butuh split DO
Distributor: 91418001 - Sales document: 1024683085 pada delivery: SAGBT-03H050009 terdapat sisa utilization sebesar 0.5765 yang tidak digunakan.
